In [ ]:
# wandb's run-file listing (run.files()) breaks when a run's file count is an exact multiple of
# the page size (default 50): the paginator requests one page past the end and the API returns
# null -> "TypeError: 'NoneType' object is not subscriptable". The loaders below swallow
# exceptions per panel, so this silently dropped insets/panels from figures. List all files in a
# single page instead (per_page far above any run's file count), falling back to page sizes that
# don't divide typical counts. Returned as a list -- all callers here only iterate over it.
from wandb.apis.public import Run as _WandbRun

if not getattr(_WandbRun.files, "_single_page_patched", False):
    _orig_run_files = getattr(_WandbRun.files, "_orig", _WandbRun.files)

    def _run_files_single_page(self, names=None, per_page=None):
        last_err = None
        for pp in (10_000, 997, 37):
            try:
                return list(_orig_run_files(self, names=names or [], per_page=pp))
            except TypeError as e:
                last_err = e
        raise RuntimeError(f"run.files() failed for {'/'.join(self.path)}") from last_err

    _run_files_single_page._single_page_patched = True
    _run_files_single_page._orig = _orig_run_files
    _WandbRun.files = _run_files_single_page


In [ ]:
from __future__ import annotations

import json
import re
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from matplotlib.axes import Axes
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


LABEL_FONTSIZE = 15
LEGEND_FONTSIZE = 11
TITLE_FONTSIZE = 14
TICK_FONTSIZE = 12

import os
import wandb as _wandb_for_entity
# W&B entity: $WANDB_ENTITY if set, otherwise your default entity.
ENTITY = os.environ.get("WANDB_ENTITY") or _wandb_for_entity.Api().default_entity
# All three Experiment-2 scripts log to the same project (configs/experiment_2_lost_latents/base.yaml);
# runs are told apart by their job_type / script, and tables by their data case.
PROJECT = "experiment_2_lost_latents"
PROJECT_A = PROJECT_B = PROJECT_C = PROJECT
SWEEP_B = None  # optionally restrict the full-sequence runs to one W&B sweep id
CASE_NAME = "english"  # main-text figures use the English adjective-noun chains
SCRIPT_A = "main_by_input_data.py"                               # a) lost rates (+ stepwise, App. F)
SCRIPT_B = "what_happened_to_lost_features_full_sequence.py"     # b)-d) main lines
SCRIPT_C = "what_happened_to_lost_features_noun.py"              # b)-d) insets
JOB_TYPE_BY_SCRIPT = {
    SCRIPT_A: "lost_rate_by_input",
    SCRIPT_B: "lost_features_full_sequence",
    SCRIPT_C: "lost_features_noun",
}


def get_run_script(run):
    jt = str(getattr(run, "job_type", "") or "")
    for script, job_type in JOB_TYPE_BY_SCRIPT.items():
        if jt == job_type:
            return script
    meta = run.metadata or {}
    program = str(meta.get("program", "") or meta.get("codePath", "") or "")
    for script in JOB_TYPE_BY_SCRIPT:
        if program.endswith(script):
            return script
    return None


KNOWN_CASES = ("english", "german", "human_typicality")


def _prefer_case(hits):
    """Runs log one table per data case under media/table/<case>/...; keep only CASE_NAME's.
    Tables without any case segment (older single-dataset runs) are kept as they are."""
    case_hits = [f for f in hits if f"/{CASE_NAME}/" in f.name]
    if case_hits:
        return case_hits
    return [f for f in hits if not any(f"/{c}/" in f.name for c in KNOWN_CASES)]


def _has_case_tables(run) -> bool:
    names = [f.name for f in run.files() if f.name.endswith(".table.json")]
    return any(f"/{CASE_NAME}/" in n for n in names) or (
        bool(names) and not any(f"/{c}/" in n for n in names for c in KNOWN_CASES))


def latest_run_with_case(runs):
    """Newest run that has CASE_NAME tables: data cases can be (re)run in separate runs,
    so the newest run of an SAE may only contain another case."""
    for run in sorted(runs, key=lambda r: str(getattr(r, "created_at", "")), reverse=True):
        if _has_case_tables(run):
            return run
    return latest_run(runs)
SETUP_NAME = "A_fixed_base"
POOLING = "last_token"
MIX_MODE_B = "lost_lasttok_signature_preact"

# choose either "aggregated" or an integer k such as 1, 2, 3, 4, 5
B_C_MODE: Any = 1  # or "aggregated"

OUTPUT_DIR = Path("wandb_model_grids")
CACHE_DIR = Path(".wandb_plot_cache")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ORDER = [
    "gemma-3-270m",
    "gemma-3-1b-pt",
    "gemma-3-4b-pt",
    "gemma-3-12b-pt",
    "gemma-3-27b-pt",
    "gpt2-small",
    "meta-llama/Llama-3.1-8B",
    "mistralai/Mistral-7B-v0.1",
]

GEMMA_MODELS = {
    "gemma-3-270m",
    "gemma-3-1b-pt",
    "gemma-3-4b-pt",
    "gemma-3-12b-pt",
    "gemma-3-27b-pt",
}
# Per-model interpretability layers (the subset available for medium SAEs).
# All SAE types for a given model are shown on these layers so color coding is
# consistent across rows. Non-medium SAEs additionally show layer 0.
GEMMA_INTERPRETABILITY_LAYERS: Dict[str, List[int]] = {
    "gemma-3-270m":  [5, 9, 12, 15],
    "gemma-3-1b-pt": [7, 13, 17, 22],
    "gemma-3-4b-pt": [9, 17, 22, 29],
    "gemma-3-12b-pt": [12, 24, 31, 41],
    "gemma-3-27b-pt": [16, 31, 40, 53],
}

EXCLUDE_FROM_MAIN = {("gemma-3-270m", "16k big")}
SPECIAL_CASES = [("gemma-3-270m", "16k big")]

BOTTOM_INLAY_MODELS_B = {
    "gemma-3-4b-pt",
    "gemma-3-12b-pt",
    "gemma-3-27b-pt",
    "meta-llama/Llama-3.1-8B",
    "mistralai/Mistral-7B-v0.1",
}


def cfg_get(cfg: Any, *keys: str, default=None):
    if not isinstance(cfg, dict):
        return default
    cur, ok = cfg, True
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            ok = False
            break
    return cur if ok else cfg.get(".".join(keys), default)


def safe_name(x: str) -> str:
    return x.replace("/", "__").replace(" ", "_")


def latest_run(runs):
    return sorted(runs, key=lambda r: str(getattr(r, "created_at", "")))[-1]


def model_sort_key(x: str) -> int:
    return MODEL_ORDER.index(x) if x in MODEL_ORDER else 10**9


def get_run_name(run) -> str:
    return str(getattr(run, "name", "") or "")


def sparse_integer_xticks(vals: List[int], n_ticks: int = 5) -> List[int]:
    vals = sorted(map(int, vals))
    if len(vals) <= n_ticks:
        return vals
    idxs = [round(i * (len(vals) - 1) / (n_ticks - 1)) for i in range(n_ticks)]
    ticks, seen = [], set()
    for idx in idxs:
        x = vals[idx]
        if x not in seen:
            ticks.append(x)
            seen.add(x)
    return ticks


def pretty_model_name(model_name: str) -> str:
    return {
        "gemma-3-270m": "Gemma 3 270M",
        "gemma-3-1b-pt": "Gemma 3 1B",
        "gemma-3-4b-pt": "Gemma 3 4B",
        "gemma-3-12b-pt": "Gemma 3 12B",
        "gemma-3-27b-pt": "Gemma 3 27B",
        "gpt2-small": "GPT-2 Small",
        "meta-llama/Llama-3.1-8B": "Llama 3.1 8B",
        "mistralai/Mistral-7B-v0.1": "Mistral 7B",
    }.get(model_name, model_name)


def get_run_text_blob(run) -> str:
    cfg = run.config or {}
    try:
        return (get_run_name(run) + " || " + json.dumps(cfg, sort_keys=True)).lower()
    except Exception:
        return (get_run_name(run) + " || " + str(cfg)).lower()


def get_model_name(run) -> Optional[str]:
    cfg = run.config or {}
    group = str(getattr(run, "group", "") or "")

    vals = [
        cfg_get(cfg, "sae", "model", "name"),
        cfg_get(cfg, "model", "name"),
        cfg_get(cfg, "sae_model_name"),
        cfg_get(cfg, "model_name"),
        group,
    ]
    for v in vals:
        if v is None:
            continue
        s = str(v)

        if s in MODEL_ORDER:
            return s

        s_low = s.lower()
        if "gemma" in s_low and "270m" in s_low:
            return "gemma-3-270m"
        if "gemma" in s_low and "1b" in s_low:
            return "gemma-3-1b-pt"
        if "gemma" in s_low and "4b" in s_low:
            return "gemma-3-4b-pt"
        if "gemma" in s_low and "12b" in s_low:
            return "gemma-3-12b-pt"
        if "gemma" in s_low and "27b" in s_low:
            return "gemma-3-27b-pt"
        if "gpt2-small" in s_low:
            return "gpt2-small"
        if "llama-3.1-8b" in s_low or "meta-llama" in s_low and "8b" in s_low:
            return "meta-llama/Llama-3.1-8B"
        if "mistral-7b" in s_low:
            return "mistralai/Mistral-7B-v0.1"

    blob_parts = [get_run_name(run), group]
    try:
        blob_parts.append(json.dumps(cfg, sort_keys=True))
    except Exception:
        blob_parts.append(str(cfg))
    blob = " || ".join(blob_parts).lower()

    if "gemma-scope-2-270m" in blob or ("270m" in blob and "gemma" in blob):
        return "gemma-3-270m"
    if "gemma-scope-2-1b" in blob or ("1b" in blob and "gemma" in blob):
        return "gemma-3-1b-pt"
    if "gemma-scope-2-4b" in blob or ("4b" in blob and "gemma" in blob):
        return "gemma-3-4b-pt"
    if "gemma-scope-2-12b" in blob or ("12b" in blob and "gemma" in blob):
        return "gemma-3-12b-pt"
    if "gemma-scope-2-27b" in blob or ("27b" in blob and "gemma" in blob):
        return "gemma-3-27b-pt"
    if "gpt2-small" in blob:
        return "gpt2-small"
    if "llama-3.1-8b" in blob or ("meta-llama" in blob and "8b" in blob):
        return "meta-llama/Llama-3.1-8B"
    if "mistral-7b" in blob:
        return "mistralai/Mistral-7B-v0.1"

    return None


def get_sae_short_label(run) -> Optional[str]:
    run_name = get_run_name(run).lower()
    if re.search(r"gpt2-small-resid-post-v\d+-128k", run_name):
        return "128k"
    if re.search(r"gpt2-small-resid-post-v\d+-32k", run_name):
        return "32k"
    if re.search(r"llama[_ ]scope[_ ]lxr[_ ]8x[_ ]l\{\}r[_ ]8x", run_name):
        return "32k"
    if re.search(r"llama[_ ]scope[_ ]lxr[_ ]32x[_ ]l\{\}r[_ ]32x", run_name):
        return "128k"  # 2**17 = 131,072 latents; called 128k by the release, like GPT-2 v5-128k
    if "mistral-7b-res-wg" in run_name:
        return "65k"  # d_sae = 65,536

    cfg = run.config or {}
    vals = [
        cfg_get(cfg, "sae", "release"),
        cfg_get(cfg, "sae_release"),
        cfg_get(cfg, "sae", "name"),
        cfg_get(cfg, "sae_name"),
        cfg.get("sae"),
        run_name,
    ]
    for v in vals:
        if v is None:
            continue
        s = str(v).lower()
        for pat in [
            r"(\d+k)_(big|small|medium|canonical)\.yaml",
            r"(\d+k)_(big|small|medium|canonical)",
            r"width[_ -](\d+k)(?:[_ -]l\d+)?[_ -](big|small|medium|canonical)\b",
            r"\b(\d+k)\s*[_ -]?\s*(big|small|medium|canonical)\b",
        ]:
            m = re.search(pat, s)
            if m:
                return f"{m.group(1)} {m.group(2)}"
        if "base.yaml" in s or re.search(r"\bbase\b", s):
            return "base"

    raw = (
        cfg_get(cfg, "sae", "release")
        or cfg_get(cfg, "sae_release")
        or cfg_get(cfg, "sae", "name")
        or cfg_get(cfg, "sae_name")
    )
    return str(raw).replace(".yaml", "").replace("_", " ").strip() if raw is not None else run_name


def get_pooling(run) -> str:
    return str(cfg_get(run.config or {}, "pooling", default=POOLING))


def select_representative_layers(layers: List[int], n_select: int = 5) -> List[int]:
    layers = sorted(map(int, layers))
    if len(layers) <= n_select:
        return layers
    idxs = [round(i * (len(layers) - 1) / (n_select - 1)) for i in range(n_select)]
    out, seen = [], set()
    for idx in idxs:
        l = layers[idx]
        if l not in seen:
            out.append(l)
            seen.add(l)
    return out


def is_medium_sae(sae_label: str) -> bool:
    return bool(re.search(r"\bmedium\b", sae_label.lower()))


def select_gemma_layers(available_layers: List[int], sae_label: str, model_name: str) -> List[int]:
    """Select layers for Gemma SAE plots so color coding is consistent across SAE types.

    All SAE types for a model plot the model's interpretability layers in the same
    order, giving them identical colors. Non-medium SAEs get layer 0 appended so it
    doesn't displace the color assignment of the shared interpretability layers.
    """
    available = set(available_layers)
    interp_layers = GEMMA_INTERPRETABILITY_LAYERS.get(model_name, [])
    interp = [l for l in interp_layers if l in available]
    if not is_medium_sae(sae_label) and 0 in available:
        interp = interp + [0]
    return interp


def sae_sort_key(x: str):
    if x == "base":
        return (999, 999, x)
    m = re.match(r"(\d+)k(?:\s+(big|small|medium))?$", x)
    if m:
        size, var = int(m.group(1)), m.group(2)
        return (size, 0 if var == "big" else 1 if var == "medium" else 2, x)
    return (998, 998, x)


def parse_table_json(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8") as f:
        obj = json.load(f)
    return pd.DataFrame(obj["data"], columns=obj["columns"])


def find_table_file(run, contains: str):
    hits = _prefer_case([f for f in run.files() if contains in f.name and f.name.endswith(".table.json")])
    return sorted(hits, key=lambda x: x.name)[0] if hits else None


def find_metric_table_file(run, metric_name: str):
    target_prefix = f"{metric_name}_data_"
    hits = []
    for f in run.files():
        if not f.name.endswith(".table.json"):
            continue
        base = Path(f.name).name
        if base.startswith(target_prefix):
            hits.append(f)
    hits = _prefer_case(hits)
    return sorted(hits, key=lambda x: x.name)[0] if hits else None


def download_file(file_obj, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    return Path(file_obj.download(root=str(out_dir), replace=True).name)


def normalize_panel_a_df(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "layer": df["layer"].astype(int),
        "k": df["k"].astype(int),
        "mean": df["mean_lost_rate"].astype(float),
        "std": df["std_lost_rate"].astype(float),
        # main_by_input_data.py logs both setups (A_fixed_base, B_stepwise) in one table, column "setup"
        "setup_name": (df["setup"] if "setup" in df.columns else df["setup_name"] if "setup_name" in df.columns
                       else pd.Series(SETUP_NAME, index=df.index)).astype(str),
        "pooling": df["pooling"].astype(str) if "pooling" in df.columns else POOLING,
    })


def normalize_project_b_metric_df(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    if "metric" in df.columns and "metric_name" not in df.columns:
        rename["metric"] = "metric_name"
    if "mix_mode" in df.columns and "pooling" not in df.columns:
        rename["mix_mode"] = "mix_mode"
    df = df.rename(columns=rename).copy()
    if "mix_mode" in df.columns:  # multi-case runs log "<case>/<mix_mode>"; compare on the mix mode only
        df["mix_mode"] = df["mix_mode"].astype(str).str.split("/").str[-1]

    df["layer"] = df["layer"].astype(int)
    df["k"] = df["k"].astype(int)
    df["mean"] = df["mean"].astype(float)
    df["std"] = df["std"].astype(float)
    df["metric_name"] = df["metric_name"].astype(str)
    df["setup_name"] = df["setup_name"].astype(str)
    df["mix_mode"] = df["mix_mode"].astype(str)
    return df


def load_panel_a_df(run) -> Optional[pd.DataFrame]:
    f = find_table_file(run, "results_table")
    if f is None:
        return None
    path = download_file(
        f,
        CACHE_DIR / safe_name(get_model_name(run) or "unknown_model") / safe_name(get_sae_short_label(run) or "unknown_sae") / "a",
    )
    return normalize_panel_a_df(parse_table_json(path))


def load_project_b_metric_df(run, metric_name: str) -> Optional[pd.DataFrame]:
    f = find_metric_table_file(run, metric_name)
    if f is None:
        return None
    path = download_file(
        f,
        CACHE_DIR / safe_name(get_model_name(run) or "unknown_model") / safe_name(get_sae_short_label(run) or "unknown_sae") / "bc_metric",
    )
    return normalize_project_b_metric_df(parse_table_json(path))

def load_project_c_metric_df(run, metric_name: str) -> Optional[pd.DataFrame]:
    f = find_metric_table_file(run, metric_name)
    if f is None:
        return None
    path = download_file(
        f,
        CACHE_DIR / safe_name(get_model_name(run) or "unknown_model") / safe_name(get_sae_short_label(run) or "unknown_sae") / "c_metric",
    )
    return normalize_project_b_metric_df(parse_table_json(path))


def df_to_plot_dicts(df: pd.DataFrame):
    df = df[(df["setup_name"] == SETUP_NAME) & (df["pooling"] == POOLING)].copy()
    mean_by_layer_by_k, std_by_layer_by_k = defaultdict(dict), defaultdict(dict)
    for _, row in df.iterrows():
        l, k = int(row["layer"]), int(row["k"])
        mean_by_layer_by_k[l][k] = float(row["mean"])
        std_by_layer_by_k[l][k] = float(row["std"])
    return dict(mean_by_layer_by_k), dict(std_by_layer_by_k), sorted(mean_by_layer_by_k.keys())


def panel_a_df_to_layer_dicts(df: pd.DataFrame, mode: Any):
    sub = df[
        (df["setup_name"] == SETUP_NAME) &
        (df["pooling"] == POOLING)
    ].copy()

    target_k = 1 if mode == "aggregated" else int(mode)
    sub = sub[sub["k"] == target_k].copy()

    if sub.empty:
        raise ValueError(f"no rows for k={target_k}")

    layers = sorted(sub["layer"].astype(int).tolist())
    mean_by_layer = {int(r["layer"]): float(r["mean"]) for _, r in sub.iterrows()}
    std_by_layer = {int(r["layer"]): float(r["std"]) for _, r in sub.iterrows()}
    return layers, mean_by_layer, std_by_layer


def metric_df_to_layer_dicts(df: pd.DataFrame, mode: Any):
    sub = df[
        (df["setup_name"] == SETUP_NAME) &
        (df["mix_mode"] == MIX_MODE_B)
    ].copy()

    if sub.empty:
        raise ValueError(f"empty table after filtering: setup={SETUP_NAME}, mix_mode={MIX_MODE_B}")

    if mode == "aggregated":
        stats = sub.groupby("layer", as_index=False).agg(mean=("mean", "mean"), std=("mean", "std"))
        stats["std"] = stats["std"].fillna(0.0)
    else:
        k = int(mode)
        sub = sub[sub["k"] == k].copy()
        if sub.empty:
            raise ValueError(f"no rows for k={k}")
        stats = sub[["layer", "mean", "std"]].copy()

    layers = sorted(stats["layer"].astype(int).tolist())
    mean_by_layer = {int(r["layer"]): float(r["mean"]) for _, r in stats.iterrows()}
    std_by_layer = {int(r["layer"]): float(r["std"]) for _, r in stats.iterrows()}
    return layers, mean_by_layer, std_by_layer


def add_panel_letter(ax, letter: str) -> None:
    ax.text(-0.10, 0.99, letter, transform=ax.transAxes, ha="right", va="top", fontsize=18, fontweight="bold", clip_on=False)


def draw_blank(ax, subtitle: str = "nothing found") -> None:
    ax.text(0.5, 0.56, subtitle, ha="center", va="center", transform=ax.transAxes, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_alpha(0.3)


def plot_against_k_on_ax(
    ax: Axes,
    selected_layers: List[int],
    mean_by_layer_by_k: Dict[int, Dict[int, float]],
    std_by_layer_by_k: Optional[Dict[int, Dict[int, float]]] = None,
    title: str = "",
    ylabel: str = "",
    k_values: Optional[List[int]] = None,
    ylim: Optional[Tuple[float, float]] = (-0.02, 1.02),
    show_legend: bool = True,
    inlay_layers: Optional[List[int]] = None,
    mean_inlay_by_layer: Optional[Dict[int, float]] = None,
    std_inlay_by_layer: Optional[Dict[int, float]] = None,
    inlay_loc: str = "upper left",
    inlay_width: str = "24%",
    inlay_height: str = "24%",
    inlay_bbox_to_anchor: Optional[Tuple[float, float, float, float]] = (0.12, 0.02, 1.0, 1.0),
    legend_loc: str = "upper right",
) -> None:
    layers = [int(x) for x in selected_layers]

    mean_int: Dict[int, Dict[int, float]] = {}
    for layer_k, d in mean_by_layer_by_k.items():
        layer = int(layer_k)
        mean_int[layer] = {int(k): float(v) for k, v in d.items()}

    std_int: Dict[int, Dict[int, float]] = {}
    if std_by_layer_by_k is not None:
        for layer_k, d in std_by_layer_by_k.items():
            layer = int(layer_k)
            std_int[layer] = {int(k): float(v) for k, v in d.items()}

    if k_values is None:
        ks = sorted({k for layer in layers for k in mean_int.get(layer, {}).keys()})
    else:
        ks = [int(k) for k in k_values]

    if not ks:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title, fontsize=TITLE_FONTSIZE)
        ax.set_xlabel("step $k$", fontsize=LABEL_FONTSIZE)
        ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
        ax.set_ylim(-0.02, 1.02)
        ax.set_yticks(np.linspace(0, 1, 11))
        return

    k_min, k_max = ks[0], ks[-1]
    xticks = [int(round(x)) for x in np.linspace(k_min, k_max, 5)]
    xticks = sorted(set(xticks))

    for layer in layers:
        x, y, s = [], [], []
        for k in ks:
            x.append(k)
            y.append(mean_int.get(layer, {}).get(k, float("nan")))
            if std_by_layer_by_k is not None:
                s.append(std_int.get(layer, {}).get(k, 0.0))

        line = ax.plot(x, y, linewidth=2.0, label=f"layer {layer}")
        line_color = line[0].get_color()

        if std_by_layer_by_k is not None and len(s) == len(y):
            lower = [yy - ss for yy, ss in zip(y, s)]
            upper = [yy + ss for yy, ss in zip(y, s)]
            ax.fill_between(x, lower, upper, color=line_color, alpha=0.18, linewidth=0)

    ax.set_xticks(xticks)
    ax.set_xlabel("step $k$", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    ax.grid(True, linewidth=0.5, alpha=0.25)

    if ylim is not None:
        y_min, y_max = ylim
    else:
        y_min, y_max = -0.02, 1.02
    ax.set_ylim(y_min, y_max)
    ax.set_yticks(np.linspace(0, 1, 11))

    if show_legend:
        ax.legend(fontsize=LEGEND_FONTSIZE, loc=legend_loc)

    if inlay_layers is not None and mean_inlay_by_layer is not None and std_inlay_by_layer is not None:
        x_in = [int(l) for l in inlay_layers]
        if x_in:
            inlay_xticks = sparse_integer_xticks(x_in, n_ticks=5)
            y_in = [float(mean_inlay_by_layer[l]) for l in x_in]
            s_in = [float(std_inlay_by_layer[l]) for l in x_in]

            axins = inset_axes(
                ax,
                width=inlay_width,
                height=inlay_height,
                loc=inlay_loc,
                bbox_to_anchor=inlay_bbox_to_anchor,
                bbox_transform=ax.transAxes,
                borderpad=0.8,
            )

            axins.patch.set_facecolor("white")
            axins.patch.set_alpha(0.72)

            line_in = axins.plot(x_in, y_in, marker="o", markersize=3, linewidth=1.6)
            color_in = line_in[0].get_color()

            lower_in = [yy - ss for yy, ss in zip(y_in, s_in)]
            upper_in = [yy + ss for yy, ss in zip(y_in, s_in)]
            axins.fill_between(x_in, lower_in, upper_in, color=color_in, alpha=0.18, linewidth=0)

            axins.set_xticks(inlay_xticks)
            axins.set_xlim(min(x_in) - 0.5, max(x_in) + 0.5)
            axins.set_yticks([0, 0.5, 1.0])
            axins.set_ylim(-0.02, 1.02)
            axins.tick_params(axis="both", labelsize=max(11, TICK_FONTSIZE - 1), pad=2)
            axins.grid(True, alpha=0.25)
            axins.margins(x=0.06, y=0.08)


def plot_main_metric_on_ax(
    ax: Axes,
    *,
    layers: List[int],
    mean_main_by_layer: Dict[int, float],
    std_main_by_layer: Dict[int, float],
    main_label: str,
    title: str,
    ylabel: str,
    show_error_bars: bool = True,
    main_color: Optional[str] = None,
    legend_loc: str = "upper right",
    show_legend: bool = True,
) -> None:
    x = [int(l) for l in layers]
    main_xticks = sparse_integer_xticks(x, n_ticks=5)

    y_main = [float(mean_main_by_layer[l]) for l in x]
    s_main = [float(std_main_by_layer[l]) for l in x]

    line_main = ax.plot(x, y_main, marker="o", markersize=5, label=main_label, color=main_color)
    color_main = line_main[0].get_color()

    if show_error_bars:
        lower_main = [yy - ss for yy, ss in zip(y_main, s_main)]
        upper_main = [yy + ss for yy, ss in zip(y_main, s_main)]
        ax.fill_between(x, lower_main, upper_main, color=color_main, alpha=0.18, linewidth=0)

    ax.set_xlabel("Layer $l$", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    ax.set_xticks(main_xticks)
    ax.set_yticks(np.linspace(0, 1, 11))
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    if show_legend:
        ax.legend(fontsize=LEGEND_FONTSIZE, loc=legend_loc, framealpha=0.9)


def plot_two_metrics_on_ax(
    ax: Axes,
    *,
    layers: List[int],
    mean_metric1_by_layer: Dict[int, float],
    std_metric1_by_layer: Dict[int, float],
    mean_metric2_by_layer: Dict[int, float],
    std_metric2_by_layer: Dict[int, float],
    metric1_label: str,
    metric2_label: str,
    title: str,
    ylabel: str,
    show_error_bars: bool = True,
    metric1_color: Optional[str] = None,
    metric2_color: Optional[str] = None,
    legend_loc: str = "upper right",
    show_legend: bool = True,
) -> None:
    x = [int(l) for l in layers]
    xticks = sparse_integer_xticks(x, n_ticks=5)

    y1 = [float(mean_metric1_by_layer[l]) for l in x]
    s1 = [float(std_metric1_by_layer[l]) for l in x]
    y2 = [float(mean_metric2_by_layer[l]) for l in x]
    s2 = [float(std_metric2_by_layer[l]) for l in x]

    line1 = ax.plot(x, y1, marker="o", markersize=5, linewidth=2.0, label=metric1_label, color=metric1_color)
    color1 = line1[0].get_color()
    if show_error_bars:
        ax.fill_between(
            x,
            [yy - ss for yy, ss in zip(y1, s1)],
            [yy + ss for yy, ss in zip(y1, s1)],
            color=color1, alpha=0.18, linewidth=0,
        )

    line2 = ax.plot(x, y2, marker="o", markersize=5, linewidth=2.0, label=metric2_label, color=metric2_color)
    color2 = line2[0].get_color()
    if show_error_bars:
        ax.fill_between(
            x,
            [yy - ss for yy, ss in zip(y2, s2)],
            [yy + ss for yy, ss in zip(y2, s2)],
            color=color2, alpha=0.18, linewidth=0,
        )

    ax.set_xlabel("Layer $l$", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.set_title(title, fontsize=TITLE_FONTSIZE)
    ax.tick_params(axis="both", labelsize=TICK_FONTSIZE)
    ax.set_xticks(xticks)
    ax.set_xlim(min(x) - 0.5, max(x) + 0.5)
    ax.set_yticks(np.linspace(0, 1, 11))
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, alpha=0.3)
    if show_legend:
        ax.legend(fontsize=LEGEND_FONTSIZE, loc=legend_loc, framealpha=0.9)


def style_bc_axis(ax, ylabel: str, last_row: bool, layers: List[int], xlabel: str = r"evaluation layer $\ell$"):
    ax.set_ylabel(ylabel, fontsize=14)
    ax.tick_params(axis="both", labelsize=12)
    ticks = sparse_integer_xticks(layers, n_ticks=5)
    ax.set_xticks(ticks)
    ax.set_xlim(min(layers) - 0.5, max(layers) + 0.5)
    ax.set_ylim(-0.02, 1.02)
    ax.set_yticks(np.linspace(0, 1, 11))
    if last_row:
        ax.set_xlabel(xlabel, fontsize=14)
    else:
        ax.set_xlabel("")
        ax.tick_params(axis="x", labelbottom=False)


def collect_runs(project: str, sweep_id: Optional[str] = None, script: Optional[str] = None):
    api = wandb.Api(timeout=60)
    filters = {"sweep": sweep_id} if sweep_id is not None else {}
    runs = list(api.runs(f"{ENTITY}/{project}", filters=filters))
    grouped = defaultdict(list)
    for run in runs:
        model_name = get_model_name(run)
        sae_label = get_sae_short_label(run)
        if model_name is None or sae_label is None:
            continue
        if script is not None and get_run_script(run) != script:
            continue
        if script == SCRIPT_A:
            pooling = get_pooling(run)
            if pooling != POOLING:
                continue
        grouped[(model_name, sae_label)].append(run)
    return {k: latest_run_with_case(v) for k, v in grouped.items()}


def load_three_metric_tables(run_b, main_metric: str, in1_metric: str, in2_metric: str, panel_name: str, loader=None):
    if loader is None:
        loader = load_project_b_metric_df
    main_df = loader(run_b, main_metric)
    in1_df = loader(run_b, in1_metric)
    in2_df = loader(run_b, in2_metric)

    if main_df is None:
        raise ValueError(f"[{panel_name}] missing table for {main_metric}")
    if in1_df is None:
        raise ValueError(f"[{panel_name}] missing table for {in1_metric}")
    if in2_df is None:
        raise ValueError(f"[{panel_name}] missing table for {in2_metric}")

    layers_main, mean_main, std_main = metric_df_to_layer_dicts(main_df, B_C_MODE)
    layers_in1, mean_in1, std_in1 = metric_df_to_layer_dicts(in1_df, B_C_MODE)
    layers_in2, mean_in2, std_in2 = metric_df_to_layer_dicts(in2_df, B_C_MODE)

    layers = sorted(set(layers_main) & set(layers_in1) & set(layers_in2))
    if not layers:
        raise ValueError(f"[{panel_name}] no overlapping layers")

    mean_main = {l: mean_main[l] for l in layers}
    std_main = {l: std_main[l] for l in layers}
    mean_in1 = {l: mean_in1[l] for l in layers}
    std_in1 = {l: std_in1[l] for l in layers}
    mean_in2 = {l: mean_in2[l] for l in layers}
    std_in2 = {l: std_in2[l] for l in layers}

    return layers, mean_main, std_main, mean_in1, std_in1, mean_in2, std_in2


def _add_layer_inlay(ax, layers, means_list, stds_list, colors, loc="upper left", context="noun", bbox_x=0.12, width="24%"):
    x = [int(l) for l in layers]
    if not x:
        return
    axins = inset_axes(ax, width=width, height="24%", loc=loc,
                       bbox_to_anchor=(bbox_x, 0.02, 1.0, 1.0), bbox_transform=ax.transAxes, borderpad=0.8)
    axins.patch.set_facecolor("white")
    axins.patch.set_alpha(0.72)
    handles = []
    for mean_dict, std_dict, color in zip(means_list, stds_list, colors):
        y = [float(mean_dict[l]) for l in x]
        s = [float(std_dict[l]) for l in x]
        line = axins.plot(x, y, marker="o", markersize=3, linewidth=1.6, color=color)
        handles.append(line[0])
        axins.fill_between(x, [yy - ss for yy, ss in zip(y, s)], [yy + ss for yy, ss in zip(y, s)],
                           color=color, alpha=0.18, linewidth=0)
    axins.set_xticks(sparse_integer_xticks(x, n_ticks=3))
    axins.set_xlim(min(x) - 0.5, max(x) + 0.5)
    axins.set_yticks([0, 0.5, 1.0])
    axins.set_ylim(-0.02, 1.02)
    axins.tick_params(axis="both", labelsize=11, pad=2)
    axins.grid(True, alpha=0.25)
    axins.margins(x=0.06, y=0.08)
    return handles


def plot_row(
    ax_a,
    ax_b,
    ax_c,
    ax_d,
    run_a,
    run_b,
    run_c=None,
    model_name: Optional[str] = None,
    inlay_loc: str = "upper left",
    legend_loc_bcd: str = "upper right",
    legend_loc_d: Optional[str] = None,
    sae_label: str = "",
    show_letters: bool = False,
    show_legend: bool = False,
    last_row: bool = False,
    show_row_label: bool = True,
):
    if show_row_label:
        ax_a.text(-0.20, 0.5, sae_label, transform=ax_a.transAxes, rotation=90, ha="center", va="center", fontsize=12, fontweight="bold", clip_on=False)

    if show_letters:
        add_panel_letter(ax_a, "a)")
        add_panel_letter(ax_b, "b)")
        add_panel_letter(ax_c, "c)")
        add_panel_letter(ax_d, "d)")

    # Panel a: lost rate vs k, with layerwise inset
    if run_a is None:
        draw_blank(ax_a)
    else:
        try:
            df_a = load_panel_a_df(run_a)
            if df_a is None or df_a.empty:
                raise ValueError("panel A table missing or empty")

            mean_a, std_a, layers_a = df_to_plot_dicts(df_a)
            if not layers_a:
                raise ValueError("panel A no layers after filtering")

            if model_name in GEMMA_MODELS:
                selected_layers = select_gemma_layers(list(mean_a.keys()), sae_label, model_name)
            else:
                selected_layers = select_representative_layers(layers_a, n_select=5)
            layers_inlay_a, mean_inlay_a, std_inlay_a = panel_a_df_to_layer_dicts(df_a, B_C_MODE)

            plot_against_k_on_ax(
                ax=ax_a,
                selected_layers=selected_layers,
                mean_by_layer_by_k={l: mean_a[l] for l in selected_layers},
                std_by_layer_by_k={l: std_a[l] for l in selected_layers},
                title="",
                ylabel=r"lost rate $\mathcal{R}$",
                ylim=(-0.02, 1.02),
                show_legend=show_legend,
                inlay_layers=layers_inlay_a,
                mean_inlay_by_layer=mean_inlay_a,
                std_inlay_by_layer=std_inlay_a,
                inlay_loc="upper left",
                inlay_width="24%",
                inlay_height="24%",
                inlay_bbox_to_anchor=(0.12, 0.02, 1.0, 1.0),
                legend_loc="upper right",
            )

            ax_a.set_ylabel(r"lost rate $\mathcal{R}$", fontsize=14)
            ax_a.tick_params(axis="both", labelsize=12)
            if last_row:
                ax_a.set_xlabel(r"step $k$", fontsize=14)
            else:
                ax_a.set_xlabel("")
                ax_a.tick_params(axis="x", labelbottom=False)

            if show_legend:
                handles, labels = ax_a.get_legend_handles_labels()
                ax_a.legend(handles, labels, fontsize=LEGEND_FONTSIZE, loc="upper right", ncol=2, columnspacing=0.8, handlelength=1.4)

        except Exception as e:
            print(f"[A] {sae_label} -> {e}")
            draw_blank(ax_a)

    if run_b is None:
        draw_blank(ax_b)
        draw_blank(ax_c)
        draw_blank(ax_d)
    else:
        # Panels b and c: previously-active data
        try:
            layers_b, mean_b_main, std_b_main, mean_b_in1, std_b_in1, mean_b_in2, std_b_in2 = load_three_metric_tables(
                run_b,
                "prev_active_among_lost",
                "inactive_last_among_prev_active_lost",
                "negative_last_among_prev_active_lost",
                "B",
            )

            # Panel b: previously active (main line only, no inset)
            plot_main_metric_on_ax(
                ax=ax_b,
                layers=layers_b,
                mean_main_by_layer=mean_b_main,
                std_main_by_layer=std_b_main,
                main_label="detected upstream (seq)",
                title="",
                ylabel="fraction",
                show_error_bars=True,
                main_color="royalblue",
                legend_loc=legend_loc_bcd,
                show_legend=show_legend,
            )
            style_bc_axis(ax_b, "fraction", last_row, layers_b)

            # Panel c: former inset of b, promoted to full panel
            plot_two_metrics_on_ax(
                ax=ax_c,
                layers=layers_b,
                mean_metric1_by_layer=mean_b_in1,
                std_metric1_by_layer=std_b_in1,
                mean_metric2_by_layer=mean_b_in2,
                std_metric2_by_layer=std_b_in2,
                metric1_label="near-zero (seq)",
                metric2_label="negative (seq)",
                title="",
                ylabel="fraction",
                show_error_bars=True,
                metric1_color="red",
                metric2_color="cyan",
                legend_loc=legend_loc_bcd,
                show_legend=show_legend,
            )
            style_bc_axis(ax_c, "fraction", last_row, layers_b, xlabel=r"evaluation layer $\ell$")

        except Exception as e:
            print(f"[B/C] {sae_label} -> {e}")
            draw_blank(ax_b)
            draw_blank(ax_c)

        if run_c is not None:
            try:
                layers_bc_c, mean_bc_c, std_bc_c, mean_bc_c1, std_bc_c1, mean_bc_c2, std_bc_c2 = load_three_metric_tables(
                    run_c, "prev_active_among_lost", "inactive_last_among_prev_active_lost", "negative_last_among_prev_active_lost", "BC_c", loader=load_project_c_metric_df
                )
                h_b = _add_layer_inlay(ax_b, layers_bc_c, [mean_bc_c], [std_bc_c], ["darkorange"], loc=inlay_loc, bbox_x=0.06, width="21%")
                h_c = _add_layer_inlay(ax_c, layers_bc_c, [mean_bc_c1, mean_bc_c2], [std_bc_c1, std_bc_c2], ["magenta", "navy"], loc=inlay_loc)
                if show_legend:
                    eh, el = ax_b.get_legend_handles_labels()
                    ax_b.legend(eh + h_b, el + ["detected upstream (noun)"], fontsize=LEGEND_FONTSIZE, loc=legend_loc_bcd, framealpha=0.9)
                    eh, el = ax_c.get_legend_handles_labels()
                    ax_c.legend(eh + h_c, el + ["near-zero (noun)", "negative (noun)"], fontsize=LEGEND_FONTSIZE, loc=legend_loc_bcd, framealpha=0.9)
            except Exception:
                pass

        # Panel d: former inset of old never-active panel, promoted to full panel
        try:
            layers_d, _, _, mean_d_in1, std_d_in1, mean_d_in2, std_d_in2 = load_three_metric_tables(
                run_b,
                "never_prev_active_among_lost",
                "always_inactive_among_never_prev_active_lost",
                "always_negative_among_never_prev_active_lost",
                "D",
            )

            plot_two_metrics_on_ax(
                ax=ax_d,
                layers=layers_d,
                mean_metric1_by_layer=mean_d_in1,
                std_metric1_by_layer=std_d_in1,
                mean_metric2_by_layer=mean_d_in2,
                std_metric2_by_layer=std_d_in2,
                metric1_label="always near-zero (seq)",
                metric2_label="always negative (seq)",
                title="",
                ylabel="fraction",
                show_error_bars=True,
                metric1_color="darkorange",
                metric2_color="green",
                legend_loc=legend_loc_d or legend_loc_bcd,
                show_legend=show_legend,
            )
            style_bc_axis(ax_d, "fraction", last_row, layers_d)

        except Exception as e:
            print(f"[D] {sae_label} -> {e}")
            draw_blank(ax_d)

        if run_c is not None:
            try:
                layers_d_c, _, _, mean_d_c1, std_d_c1, mean_d_c2, std_d_c2 = load_three_metric_tables(
                    run_c, "never_prev_active_among_lost", "always_inactive_among_never_prev_active_lost", "always_negative_among_never_prev_active_lost", "D_c", loader=load_project_c_metric_df
                )
                h_d = _add_layer_inlay(ax_d, layers_d_c, [mean_d_c1, mean_d_c2], [std_d_c1, std_d_c2], ["crimson", "purple"], loc="upper left", bbox_x=0.06)
                if show_legend:
                    eh, el = ax_d.get_legend_handles_labels()
                    ax_d.legend(eh + h_d, el + ["always near-zero (noun)", "always negative (noun)"], fontsize=LEGEND_FONTSIZE, loc=legend_loc_d or legend_loc_bcd, framealpha=0.9)
            except Exception:
                pass


def make_main_grids(runs_a, runs_b, runs_c):
    model_to_saes = defaultdict(list)
    for model_name, sae_label in runs_a.keys():
        if (model_name, sae_label) not in EXCLUDE_FROM_MAIN:
            model_to_saes[model_name].append(sae_label)

    for model_name in sorted(model_to_saes.keys(), key=model_sort_key):
        sae_labels = sorted(set(model_to_saes[model_name]), key=sae_sort_key)
        print(f"  {pretty_model_name(model_name)}: {sae_labels}")
        n_rows = len(sae_labels)

        fig, axes = plt.subplots(n_rows, 4, figsize=(21.0, max(2.6 * n_rows, 4.3)), squeeze=False)
        fig.suptitle(pretty_model_name(model_name), fontsize=20, y=0.985)

        for i, sae_label in enumerate(sae_labels):
            _inlay_loc = "center left" if model_name == "mistralai/Mistral-7B-v0.1" else "upper left"
            _legend_loc_bcd = "center right" if model_name == "mistralai/Mistral-7B-v0.1" else "upper right"
            _legend_loc_d = "lower right" if model_name == "gemma-3-27b-pt" else None
            plot_row(
                axes[i, 0],
                axes[i, 1],
                axes[i, 2],
                axes[i, 3],
                runs_a.get((model_name, sae_label)),
                runs_b.get((model_name, sae_label)),
                runs_c.get((model_name, sae_label)),
                model_name=model_name,
                inlay_loc=_inlay_loc,
                legend_loc_bcd=_legend_loc_bcd,
                legend_loc_d=_legend_loc_d,
                sae_label=sae_label,
                show_letters=(i == 0),
                show_legend=(i == 0),
                last_row=(i == n_rows - 1),
                show_row_label=True,
            )

        fig.subplots_adjust(left=0.07, right=0.995, bottom=0.06, top=0.93, hspace=0.15, wspace=0.24)
        suffix = f"__bc_mode_{B_C_MODE}" if B_C_MODE != "aggregated" else "__bc_mode_aggregated"
        out_path = OUTPUT_DIR / f"{safe_name(model_name)}{suffix}__grid.png"
        fig.savefig(out_path, dpi=220, bbox_inches="tight")
        plt.close(fig)
        print(f"wrote {out_path}")


def make_special_case_figures(runs_a, runs_b, runs_c):
    for model_name, sae_label in SPECIAL_CASES:
        run_a = runs_a.get((model_name, sae_label))
        run_b = runs_b.get((model_name, sae_label))
        if run_a is None and run_b is None:
            continue

        fig, axes = plt.subplots(1, 4, figsize=(19.0, 4.0), squeeze=False)
        plot_row(
            axes[0, 0],
            axes[0, 1],
            axes[0, 2],
            axes[0, 3],
            run_a,
            run_b,
            runs_c.get((model_name, sae_label)),
            model_name=model_name,
            sae_label=sae_label,
            show_letters=True,
            show_legend=True,
            last_row=True,
            show_row_label=False,
        )

        fig.subplots_adjust(left=0.05, right=0.995, bottom=0.16, top=0.93, hspace=0.1, wspace=0.24)
        suffix = f"__bc_mode_{B_C_MODE}" if B_C_MODE != "aggregated" else "__bc_mode_aggregated"
        out_path = OUTPUT_DIR / f"{safe_name(model_name)}__{safe_name(sae_label)}{suffix}__single.png"
        fig.savefig(out_path, dpi=220, bbox_inches="tight")
        plt.close(fig)
        print(f"wrote {out_path}")


def main():
    runs_a = collect_runs(PROJECT_A, script=SCRIPT_A)
    runs_b = collect_runs(PROJECT_B, SWEEP_B, script=SCRIPT_B)
    runs_c = collect_runs(PROJECT_C, script=SCRIPT_C)

    print("models in project A:", sorted({k[0] for k in runs_a.keys()}, key=model_sort_key))
    print("models in project B:", sorted({k[0] for k in runs_b.keys()}, key=model_sort_key))
    print("models in project C:", sorted({k[0] for k in runs_c.keys()}, key=model_sort_key))
    print("B/C mode:", B_C_MODE)

    make_main_grids(runs_a, runs_b, runs_c)
    make_special_case_figures(runs_a, runs_b, runs_c)


main()


In [ ]:
# -- Fetch experiment_1_set_consistency_modeB --
PROJECT_MODEB   = PROJECT  # stepwise (B_stepwise) tables are logged by main_by_input_data.py
CACHE_DIR_MODEB = CACHE_DIR / "modeb"
CACHE_DIR_MODEB.mkdir(parents=True, exist_ok=True)

PRETTY_MODEL_MB   = {m: pretty_model_name(m) for m in MODEL_ORDER}
ROWLABEL_FONTSIZE = LABEL_FONTSIZE

_api_mb  = wandb.Api(timeout=60)
_mb_runs = [r for r in _api_mb.runs(f"{ENTITY}/{PROJECT_MODEB}") if get_run_script(r) == SCRIPT_A]

_wrun_groups_mb: Dict[Tuple[str, str], list] = defaultdict(list)
for _run in _mb_runs:
    _mn = get_model_name(_run)
    _sl = get_sae_short_label(_run)
    if _mn and _sl:
        _wrun_groups_mb[(_mn, _sl)].append(_run)

model_to_runs_mb: Dict[str, Dict[str, Any]] = defaultdict(dict)
for (_mn, _sl), _grp in _wrun_groups_mb.items():
    model_to_runs_mb[_mn][_sl] = latest_run_with_case(_grp)


def _dl_modeb_lostrate(run) -> Optional[pd.DataFrame]:
    hits = _prefer_case([f for f in run.files()
            if "B_stepwise" in f.name and f.name.endswith(".table.json")])
    if not hits:
        return None
    fobj = sorted(hits, key=lambda x: x.name)[-1]
    out_dir = (CACHE_DIR_MODEB
               / safe_name(get_model_name(run) or "unknown")
               / safe_name(get_sae_short_label(run) or "unknown"))
    return parse_table_json(download_file(fobj, out_dir))


model_to_mb: Dict[str, Dict[str, pd.DataFrame]] = defaultdict(dict)
for (_mn, _sl), _grp in _wrun_groups_mb.items():
    _best = latest_run_with_case(_grp)
    _df = _dl_modeb_lostrate(_best)
    if _df is not None:
        model_to_mb[_mn][_sl] = _df


def _mb_df_to_plot_dicts(df: pd.DataFrame):
    sub = df[df["setup_name"] == "B_stepwise"].copy()
    mean_by_l_k: Dict[int, Dict[int, float]] = defaultdict(dict)
    std_by_l_k:  Dict[int, Dict[int, float]] = defaultdict(dict)
    for _, row in sub.iterrows():
        l, k = int(row["layer"]), int(row["k"])
        mean_by_l_k[l][k] = float(row["mean"])
        std_by_l_k[l][k]  = float(row["std"])
    return dict(mean_by_l_k), dict(std_by_l_k), sorted(mean_by_l_k.keys())


def _mb_df_to_layer_dicts(df: pd.DataFrame, target_k: int = 1):
    sub = df[(df["setup_name"] == "B_stepwise") & (df["k"] == target_k)].copy()
    if sub.empty:
        return [], {}, {}
    layers = sorted(sub["layer"].astype(int).tolist())
    mean_by_layer = {int(r["layer"]): float(r["mean"]) for _, r in sub.iterrows()}
    std_by_layer  = {int(r["layer"]): float(r["std"])  for _, r in sub.iterrows()}
    return layers, mean_by_layer, std_by_layer


# -- Grid: models (rows) x SAE types (cols), each subplot = panel a style --

model_sae_labels_mb = {
    m: sorted(model_to_runs_mb[m].keys(), key=sae_sort_key)
    for m in MODEL_ORDER if m in model_to_runs_mb
}
n_rows_mb = len(MODEL_ORDER)
n_cols_mb = max((len(v) for v in model_sae_labels_mb.values()), default=1)
print(f'Grid: {n_rows_mb} rows x {n_cols_mb} cols')
for _m in MODEL_ORDER:
    if _m in model_sae_labels_mb:
        _has = sorted(model_to_mb.get(_m, {}).keys(), key=sae_sort_key)
        print(f'  {PRETTY_MODEL_MB.get(_m, _m)}: {model_sae_labels_mb[_m]}  [data: {_has}]')

fig_w_mb = max(5.0, n_cols_mb * 3.2) + 1.5
fig_h_mb = n_rows_mb * 3.2
fig_mb, axes_mb = plt.subplots(n_rows_mb, n_cols_mb, figsize=(fig_w_mb, fig_h_mb), squeeze=False)

for row_idx, model_name in enumerate(MODEL_ORDER):
    pretty  = PRETTY_MODEL_MB.get(model_name, model_name)
    mb_data = model_to_mb.get(model_name, {})

    axes_mb[row_idx, 0].annotate(
        pretty,
        xy=(0, 0.5), xycoords='axes fraction',
        xytext=(-0.28, 0.5), textcoords='axes fraction',
        ha='center', va='center', rotation=90,
        fontsize=ROWLABEL_FONTSIZE - 4, fontweight='bold',
        annotation_clip=False,
    )

    if model_name not in model_sae_labels_mb:
        for ax in axes_mb[row_idx]:
            ax.axis('off')
        continue

    sae_labels_mb_row = model_sae_labels_mb[model_name]
    row_data_axes  = []
    row_active_cols = []
    is_last_row    = (row_idx == n_rows_mb - 1)

    for col_idx, sae_label in enumerate(sae_labels_mb_row):
        ax = axes_mb[row_idx, col_idx]
        df = mb_data.get(sae_label)
        if df is None:
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_alpha(0.3)
            ax.set_facecolor('#f8f8f8')
            continue

        mean_by_l_k, std_by_l_k, layers = _mb_df_to_plot_dicts(df)
        if not layers:
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_alpha(0.3)
            ax.set_facecolor('#f8f8f8')
            continue

        row_active_cols.append(col_idx)
        row_data_axes.append(ax)

        if model_name in GEMMA_MODELS:
            selected_layers = select_gemma_layers(layers, sae_label, model_name)
        else:
            selected_layers = select_representative_layers(layers, n_select=5)
        _max_k = max(k for ks in mean_by_l_k.values() for k in ks.keys())
        layers_inlay, mean_inlay, std_inlay = _mb_df_to_layer_dicts(df, target_k=_max_k)

        plot_against_k_on_ax(
            ax=ax,
            selected_layers=selected_layers,
            mean_by_layer_by_k={l: mean_by_l_k[l] for l in selected_layers},
            std_by_layer_by_k={l: std_by_l_k[l]  for l in selected_layers},
            title="",
            ylabel="",
            ylim=(-0.02, 1.02),
            show_legend=(col_idx == 0),
            inlay_layers=layers_inlay,
            mean_inlay_by_layer=mean_inlay,
            std_inlay_by_layer=std_inlay,
            inlay_loc="upper left",
            inlay_width="24%",
            inlay_height="24%",
            inlay_bbox_to_anchor=(0.12, 0.02, 1.0, 1.0),
            legend_loc="upper right",
        )

        ax.set_title(sae_label, fontsize=10, pad=3)
        ax.tick_params(axis="both", labelsize=9)
        if col_idx == 0:
            ax.set_ylabel(r"lost rate $\mathcal{R}$", fontsize=11)
            ax.set_yticklabels([f'{v:.1f}' for v in np.linspace(0, 1, 11)], fontsize=9)
        else:
            ax.set_ylabel("")
            ax.set_yticklabels([])

        ax.set_xlabel(r"step $k$", fontsize=10)
        ax.tick_params(axis="x", labelbottom=True)

    for col_idx in range(len(sae_labels_mb_row), n_cols_mb):
        axes_mb[row_idx, col_idx].axis('off')

fig_mb.subplots_adjust(left=0.15, right=0.99, top=0.98, bottom=0.04,
                       hspace=0.35, wspace=0.08)

out_mb = OUTPUT_DIR / 'experiment_modeb_lostrate_by_k.png'
fig_mb.savefig(out_mb, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig_mb)
print(f'Saved: {out_mb}')


In [ ]:
# ── K-sensitivity of mechanistic quantities (panels b–d) ────────────────────
# For each model / SAE variant / layer compute:
#   range_l = max_k q(l,k) – min_k q(l,k)
# then average over layers: k_sensitivity(M,S) = mean_l range_l
#
# Low values mean the quantity changes little with k, so k=1 is representative.
#
# Caption:
# "K-sensitivity of the mechanistic quantities reported in panels (b)–(d) of
# the lost-latent sweep. For each model, SAE variant, and measured quantity,
# we compute the range across k at each layer and average this range over
# layers. Each point is one model/SAE variant. Main panels show full-sequence
# quantities; insets show the corresponding noun-only quantities. Low values
# indicate weak k-dependence, supporting k=1 as a representative setting."

from matplotlib.lines import Line2D

_MODEL_COLORS_KS: Dict[str, str] = {
    "gemma-3-270m":              "#e377c2",
    "gemma-3-1b-pt":             "#1f77b4",
    "gemma-3-4b-pt":             "#ff7f0e",
    "gemma-3-12b-pt":            "#2ca02c",
    "gemma-3-27b-pt":            "#d62728",
    "gpt2-small":                "#9467bd",
    "meta-llama/Llama-3.1-8B":   "#8c564b",
    "mistralai/Mistral-7B-v0.1": "#7f7f7f",
}
_SAE_MARKERS_KS: Dict[str, str] = {
    "16k big":    "X",
    "16k medium": "^",
    "16k small":  "o",
    "262k big":   "P",
    "262k medium": "D",
    "262k small":  "s",
    "128k":        "*",
    "32k":         "p",
    "65k":         "v",
}


def _ksens_from_df(df: pd.DataFrame) -> Optional[float]:
    """mean_l ( max_k q(l,k) – min_k q(l,k) )"""
    sub = df[
        (df["setup_name"] == SETUP_NAME) &
        (df["mix_mode"] == MIX_MODE_B)
    ]
    if sub.empty:
        return None
    ranges = [
        float(g["mean"].max() - g["mean"].min())
        for _, g in sub.groupby("layer")
    ]
    return float(np.mean(ranges)) if ranges else None


# (metric_name, "B" = full-seq project B,  "C" = noun-only project C)
_KS_SPEC: List[Tuple[str, str]] = [
    ("prev_active_among_lost",                       "B"),
    ("prev_active_among_lost",                       "C"),
    ("inactive_last_among_prev_active_lost",         "B"),
    ("inactive_last_among_prev_active_lost",         "C"),
    ("negative_last_among_prev_active_lost",         "B"),
    ("negative_last_among_prev_active_lost",         "C"),
    ("always_inactive_among_never_prev_active_lost", "B"),
    ("always_inactive_among_never_prev_active_lost", "C"),
    ("always_negative_among_never_prev_active_lost", "B"),
    ("always_negative_among_never_prev_active_lost", "C"),
]

_ks_runs_b = collect_runs(PROJECT_B, SWEEP_B, script=SCRIPT_B)
_ks_runs_c = collect_runs(PROJECT_C, script=SCRIPT_C)

_ks_records: List[Dict] = []
for (_mn, _sl), _rb in _ks_runs_b.items():
    _rc = _ks_runs_c.get((_mn, _sl))
    for _metric, _src in _KS_SPEC:
        try:
            if _src == "B":
                _df = load_project_b_metric_df(_rb, _metric)
            else:
                _df = load_project_c_metric_df(_rc, _metric) if _rc else None
            if _df is None or _df.empty:
                continue
            _ks = _ksens_from_df(_df)
            if _ks is not None:
                _ks_records.append({"model": _mn, "sae": _sl,
                                    "metric": _metric, "source": _src, "ks": _ks})
        except Exception as _e:
            print(f"[ksens] {_mn}/{_sl}/{_metric}/{_src}: {_e}")

_ks_df = pd.DataFrame(_ks_records)
print(f"Collected {len(_ks_df)} records  "
      f"({_ks_df[['model','sae']].drop_duplicates().shape[0]} model/SAE pairs)")
print(_ks_df.groupby(["source", "metric"])["ks"]
      .agg(["count", "mean", "max"]).round(3).to_string())


def _draw_ksens_ax(
    ax: Axes,
    groups: List[Tuple[str, str, str]],   # (metric, source, x_label)
    df: pd.DataFrame,
    ms: int = 60,
    jitter: float = 0.13,
    rng: Optional[np.random.Generator] = None,
    tick_fs: int = 10,
) -> List[float]:
    """Boxplot + jittered scatter for k-sensitivity. Returns all plotted y-values."""
    if rng is None:
        rng = np.random.default_rng(0)
    all_y: List[float] = []
    for xi, (metric, src, lbl) in enumerate(groups):
        sub = df[(df["metric"] == metric) & (df["source"] == src)]
        vals = sub["ks"].dropna().values
        if len(vals) == 0:
            continue
        all_y.extend(vals.tolist())
        ax.boxplot(
            [vals], positions=[xi], widths=0.35,
            patch_artist=True, showfliers=False,
            medianprops=dict(color="black", linewidth=1.4),
            boxprops=dict(facecolor="lightgray", alpha=0.45, linewidth=0.7),
            whiskerprops=dict(linewidth=0.7),
            capprops=dict(linewidth=0.7),
        )
        for _, row in sub.iterrows():
            ax.scatter(
                xi + rng.uniform(-jitter, jitter), row["ks"],
                color=_MODEL_COLORS_KS.get(row["model"], "black"),
                marker=_SAE_MARKERS_KS.get(row["sae"], "o"),
                s=ms, alpha=0.85, linewidths=0.4,
                edgecolors="white", zorder=3,
            )
    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels([g[2] for g in groups], fontsize=tick_fs)
    ax.set_xlim(-0.5, len(groups) - 0.5)
    ax.grid(axis="y", linewidth=0.4, alpha=0.3)
    ax.set_ylim(bottom=0)
    return all_y


_ks_panels: List[Tuple] = [
    # (main_groups, inset_groups, panel_letter)
    (
        [("prev_active_among_lost",                       "B", "prev.\nactive")],
        [("prev_active_among_lost",                       "C", "prev.\nactive")],
        "a)",
    ),
    (
        [("inactive_last_among_prev_active_lost",         "B", "near-zero"),
         ("negative_last_among_prev_active_lost",         "B", "negative")],
        [("inactive_last_among_prev_active_lost",         "C", "near-zero"),
         ("negative_last_among_prev_active_lost",         "C", "negative")],
        "b)",
    ),
    (
        [("always_inactive_among_never_prev_active_lost", "B", "always\nnear-zero"),
         ("always_negative_among_never_prev_active_lost", "B", "always\nnegative")],
        [("always_inactive_among_never_prev_active_lost", "C", "always\nnear-zero"),
         ("always_negative_among_never_prev_active_lost", "C", "always\nnegative")],
        "c)",
    ),
]

_fig_ks, _axes_ks = plt.subplots(1, 3, figsize=(15, 5))
_rng_ks = np.random.default_rng(42)
_all_y_ks: List[float] = []
_axins_ks: List[Axes] = []

for _ax, (_main_g, _ins_g, _ltr) in zip(_axes_ks, _ks_panels):
    _all_y_ks.extend(_draw_ksens_ax(_ax, _main_g, _ks_df, rng=_rng_ks))
    _ax.set_ylabel("k-sensitivity", fontsize=12)
    _ax.tick_params(axis="both", labelsize=10)
    _ax.text(-0.14, 1.06, _ltr, transform=_ax.transAxes,
             ha="left", va="bottom", fontsize=14, fontweight="bold")

    _axi = inset_axes(
        _ax, width="38%", height="46%", loc="upper right",
        bbox_to_anchor=(0.0, 0.0, 0.97, 0.97),
        bbox_transform=_ax.transAxes, borderpad=0,
    )
    _axi.patch.set_facecolor("white")
    _axi.patch.set_alpha(0.85)
    _all_y_ks.extend(_draw_ksens_ax(_axi, _ins_g, _ks_df, ms=28, jitter=0.09,
                                    rng=_rng_ks, tick_fs=7))
    _axi.set_ylabel("")
    _axi.tick_params(axis="both", labelsize=7)
    _axins_ks.append(_axi)

# Consistent y-axis across all panels and insets
if _all_y_ks:
    _ym_ks = min(float(np.max(_all_y_ks)) * 1.18, 0.55)
    _ym_ks = max(_ym_ks, 0.05)
    for _ax in _axes_ks:
        _ax.set_ylim(0, _ym_ks)
        _ax.set_yticks(np.round(np.linspace(0, _ym_ks, 6), 3))
    for _axi in _axins_ks:
        _axi.set_ylim(0, _ym_ks)
        _axi.set_yticks(np.round(np.linspace(0, _ym_ks, 4), 3))

# Legends placed to the right of the last panel
_leg_model_ks = _fig_ks.legend(
    handles=[
        Line2D([0], [0], marker="o", color="w",
               markerfacecolor=c, markersize=9, label=pretty_model_name(m))
        for m, c in _MODEL_COLORS_KS.items()
    ],
    title="Model", fontsize=9, title_fontsize=10, framealpha=0.9,
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0), bbox_transform=_axes_ks[-1].transAxes,
)
_fig_ks.add_artist(_leg_model_ks)
_fig_ks.legend(
    handles=[
        Line2D([0], [0], marker=mk, color="gray",
               markersize=9, linestyle="none", label=sl)
        for sl, mk in _SAE_MARKERS_KS.items()
    ],
    title="SAE variant", fontsize=9, title_fontsize=10, framealpha=0.9,
    loc="lower left",
    bbox_to_anchor=(1.01, 0.0), bbox_transform=_axes_ks[-1].transAxes,
)

_fig_ks.subplots_adjust(left=0.07, right=0.79, bottom=0.14, top=0.92, wspace=0.32)
_out_ks = OUTPUT_DIR / "k_sensitivity_panels_bcd.png"
_fig_ks.savefig(_out_ks, dpi=220, bbox_inches="tight")
plt.show()
plt.close(_fig_ks)
print(f"Wrote {_out_ks}")


In [ ]:
# ── Figure: Lost-latent workflow Sankey diagram (illustrative values) ────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.path as mpath
from collections import defaultdict

# Flow values (illustrative, normalized to 100)
TOTAL   = 100
SCALE   = 0.30    # axes-fraction height for a node with flow == TOTAL
NODE_W  = 0.022   # node width (axes fraction)
GAP     = 0.010   # gap between node edge and ribbon


def _h(f):
    return f / TOTAL * SCALE


# ── Node definitions ──────────────────────────────────────────────────────────
# {id: (cx, cy, flow, facecolor, title)}
# Columns at x = 0.07, 0.27, 0.49, 0.72
# Leaf nodes for prev_active branch: pre-activation at layer l
# Leaf nodes for never_active branch: criterion across all layers <= l
_NODES = {
    "active":       (0.07, 0.50, 100, "#7a7a7a",
                     "Active on\nbase noun"),
    "preserved":    (0.27, 0.74,  55, "#c0c0c0",
                     "Preserved on\ncomposed noun"),
    "lost":         (0.27, 0.40,  45, "#c97240",
                     "Lost on\ncomposed noun"),
    "prev_active":  (0.49, 0.52,  25, "#4e7db5",
                     "Previously active\nupstream"),
    "never_active": (0.49, 0.25,  20, "#7c6ea8",
                     "Never active\nupstream"),
    # Previously-active leaves: pre-activation evaluated AT layer l
    "nz_pa":        (0.72, 0.66,  13, "#96b8d8",
                     "Near-zero\nat layer ℓ"),
    "neg_pa":       (0.72, 0.53,   4, "#8e4040",
                     "Negative\nat layer ℓ"),
    "other_pa":     (0.72, 0.42,   8, "#909090",
                     "Other\n(at layer ℓ)"),
    # Never-active leaves: criterion assessed ACROSS all layers <= l
    "nz_na":        (0.72, 0.31,   5, "#96b8d8",
                     "Always near-zero\n(all layers ≤ ℓ)"),
    "neg_na":       (0.72, 0.20,   2, "#8e4040",
                     "Always negative\n(all layers ≤ ℓ)"),
    "other_na":     (0.72, 0.10,  13, "#909090",
                     "Other / mixed\n(all layers ≤ ℓ)"),
}

# ── Flow definitions ──────────────────────────────────────────────────────────
_FLOWS = [
    ("active",       "preserved",    55, "#c0c0c0"),
    ("active",       "lost",         45, "#c97240"),
    ("lost",         "prev_active",  25, "#4e7db5"),
    ("lost",         "never_active", 20, "#7c6ea8"),
    ("prev_active",  "nz_pa",        13, "#96b8d8"),
    ("prev_active",  "neg_pa",        4, "#8e4040"),
    ("prev_active",  "other_pa",      8, "#b0b0b0"),
    ("never_active", "nz_na",         5, "#96b8d8"),
    ("never_active", "neg_na",        2, "#8e4040"),
    ("never_active", "other_na",     13, "#b0b0b0"),
]

# ── Compute node y extents centred at cy ─────────────────────────────────────
_ext = {nid: (cy - _h(f) / 2, cy + _h(f) / 2)
        for nid, (cx, cy, f, color, title) in _NODES.items()}

# ── Allocate ribbon y-slices at source and target edges ──────────────────────
_out = defaultdict(list)
_inn = defaultdict(list)
for i, (src, tgt, *_) in enumerate(_FLOWS):
    _out[src].append(i)
    _inn[tgt].append(i)

for k in _out:
    _out[k].sort(key=lambda i: _NODES[_FLOWS[i][1]][1], reverse=True)
for k in _inn:
    _inn[k].sort(key=lambda i: _NODES[_FLOWS[i][0]][1], reverse=True)

_src_y, _tgt_y = {}, {}
for nid, idxs in _out.items():
    cur = _ext[nid][1]
    for i in idxs:
        dh = _h(_FLOWS[i][2])
        _src_y[i] = (cur - dh, cur)
        cur -= dh

for nid, idxs in _inn.items():
    cur = _ext[nid][1]
    for i in idxs:
        dh = _h(_FLOWS[i][2])
        _tgt_y[i] = (cur - dh, cur)
        cur -= dh


def _ribbon(ax, x0, yb0, yt0, x1, yb1, yt1, fc, alpha):
    mid = (x0 + x1) * 0.5
    verts = [
        (x0, yt0),
        (mid, yt0), (mid, yt1), (x1, yt1),
        (x1, yb1),
        (mid, yb1), (mid, yb0), (x0, yb0),
        (x0, yt0),
    ]
    codes = [
        mpath.Path.MOVETO,
        mpath.Path.CURVE4, mpath.Path.CURVE4, mpath.Path.CURVE4,
        mpath.Path.LINETO,
        mpath.Path.CURVE4, mpath.Path.CURVE4, mpath.Path.CURVE4,
        mpath.Path.CLOSEPOLY,
    ]
    ax.add_patch(mpatches.PathPatch(
        mpath.Path(verts, codes), fc=fc, ec="none", alpha=alpha, zorder=1,
    ))


# ── Figure ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 8.5))
ax.set_xlim(0, 1)
ax.set_ylim(-0.14, 1)
ax.axis("off")
fig.patch.set_facecolor("white")

# Draw ribbons
for i, (src, tgt, val, color) in enumerate(_FLOWS):
    x0 = _NODES[src][0] + NODE_W / 2 + GAP
    x1 = _NODES[tgt][0] - NODE_W / 2 - GAP
    alpha = 0.25 if tgt == "preserved" else 0.52
    _ribbon(ax, x0, *_src_y[i], x1, *_tgt_y[i], color, alpha)

# Draw nodes
for nid, (cx, cy, flow, color, title) in _NODES.items():
    yb, yt = _ext[nid]
    ax.add_patch(mpatches.FancyBboxPatch(
        (cx - NODE_W / 2, yb), NODE_W, yt - yb,
        boxstyle="round,pad=0.003",
        fc=color, ec="#cccccc", lw=0.8, zorder=3,
    ))
    lx = cx
    ax.text(lx, yb - 0.013, title,
            fontsize=16, fontweight="semibold", color="#111111",
            ha="center", va="top", clip_on=False, zorder=5)

# Column header labels — black, bold, larger
for cx, lbl in [
    (0.07, "Base noun\nsignature"),
    (0.27, "Preserved\nvs. lost"),
    (0.49, "Upstream\nstatus"),
    (0.72, "Pre-activation\nregime"),
]:
    ax.text(cx, 0.95, lbl, ha="center", va="top",
            fontsize=17, fontweight="bold", color="#000000", zorder=5)

plt.tight_layout(pad=0.2)
_sankey_png = OUTPUT_DIR / "lost_latent_workflow_sankey.png"
_sankey_pdf = OUTPUT_DIR / "lost_latent_workflow_sankey.pdf"
fig.savefig(_sankey_png, dpi=300, bbox_inches="tight")
fig.savefig(_sankey_pdf, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {_sankey_png}")
print(f"Saved: {_sankey_pdf}")
